In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import TimestampType, IntegerType, StringType

In [0]:
# Read the taxi zone lookup data from the CSV file inside the volume
# We explicitly set 'header' to True so Spark uses the first row as column names
df = spark.read.format('csv').option('header', True).load("/Volumes/nyctaxi/00_landing/data_sources/lookup/taxi_zone_lookup.csv")

In [0]:
# Standardize column names to snake_case, cast data types and initialize effective/end dates for historical data tracking (SCD Type 2)
# SCD Type 2 (Slowly Changing Dimensions Type 2)
# end_date = lit(None) -> The None value represents NULL. This information is currently active, and we do not yet know when its validity will expire (it is valid indefinitely).
df = df.select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone"),
    current_timestamp().cast(TimestampType()).alias("effective_date"),
    lit(None).cast(TimestampType()).alias("end_date")
)

In [0]:
# Fixed point-in-time used to "close" any changed active records
# Using a Python timestamp ensures the exact same value is written and can be referenced if needed
end_timestamp = datetime.now()

# Load the SCD2 Delta table
dt = DeltaTable.forName(spark, "nyctaxi.02_silver.taxi_zone_lookup")

In [0]:
# PASS 1: Close active records for modified attributes (SCD Type 2)
# 1. Identify active target records (end_date IS NULL) with the same business key (location_id).
# 2. Check if any tracked columns (borough, zone, service_zone) have changed compared to the source data.
# 3. If a change is detected, set end_date to the current timestamp to retire the old version.

dt.alias("t").\
    merge(
        source = df.alias("s"),
        condition = "t.location_id = s.location_id AND t.end_date IS NULL AND (t.borough != s.borough OR t.zone != s.zone OR t.service_zone != s.service_zone)").\
    whenMatchedUpdate(
        set = {
            "t.end_date": lit(end_timestamp).cast(TimestampType())
        }
    ).\
    execute()

In [0]:
# PASS 2: Insert new current versions (SCD Type 2)
# The goal is to insert the new active versions of records that were updated (closed in PASS 1) 
# and any brand-new records that do not yet exist in the target table.

# Collect the list of business keys (location_id) that were retired in PASS 1 
# to ensure they are correctly re-inserted as new active versions in the target table.
insert_id_list = [row.location_id for row in dt.toDF().filter(f"end_date = '{end_timestamp}'").select("location_id").collect()]

# Validate if there are any records to process; if the list is empty, skip the merge operation 
# to prevent syntax errors and avoid unnecessary computation
if len(insert_id_list) == 0:
    print("No updated records to insert")
else:
    dt.alias("t").\
        merge(
            source = df.alias("s"),
            condition = f"s.location_id not in ({','.join(map(str, insert_id_list))})"
        ).\
        whenNotMatchedInsert(
            values = {
                "t.location_id":"s.location_id",
                "t.borough":"s.borough",
                "t.zone":"s.zone",
                "t.service_zone":"s.service_zone",
                "t.effective_date":current_timestamp(),
                "t.end_date":lit(None).cast(TimestampType())
            }
        ).\
        execute()


In [0]:
# PASS 3: Insert brand-new records (final safety check)
# This step identifies records that are completely missing from the target table.
# It ensures that any brand-new location_ids found in the source are added as 
# the initial active version.
dt.alias("t").\
    merge(
        source = df.alias("s"),
        condition = "t.location_id = s.location_id"
    ).\
    whenNotMatchedInsert(
        values = {
            "t.location_id":"s.location_id",
            "t.borough":"s.borough",
            "t.zone":"s.zone",
            "t.service_zone":"s.service_zone",
            "t.effective_date":current_timestamp(),
            "t.end_date":lit(None).cast(TimestampType())
        }
    ).\
    execute()